# Multi-Agent and Cross-Server Attack Paths

Build and interrogate a provenance-backed attack graph for a synthetic enterprise support platform. The central distinction is **topology ≠ current executability ≠ observed effect**.

## Safety and scope

Everything is deterministic, synthetic, and offline. No remote agent, MCP server, scanner, credential, or production system is contacted. The notebook imports the canonical `lab.py`; it does not duplicate the security implementation.

## Objectives

By the end, you will verify graph provenance, distinguish observed/influence/authorized/effect edges, enumerate bounded paths, enforce current controls, measure sensitive exposure and false blocks, verify effects independently, and prove containment after propagation.

In [ ]:
from dataclasses import replace
from datetime import timedelta
from pathlib import Path
import runpy

lab = runpy.run_path(Path('lab.py'))
environment = lab['build_environment']()
graph = environment.graph
unsafe = environment.unsafe_controls
hardened = environment.hardened_controls
len(graph.assets), len(graph.edges), graph.snapshot.epoch

## 1. Inspect the typed graph

Nodes carry tenant, zone, sensitivity, audience, artifact, and workload facts. Edges preserve relation, action, exact resource, credential, registry, gateway, technique, and evidence references. This is more informative than a generic `CAN_REACH` relation.

In [ ]:
[(a.asset_id, a.kind.value, a.tenant, a.sensitivity.value, a.criticality)
 for a in graph.assets.values()]

In [ ]:
[(e.edge_id, e.source, e.relation.value, e.target, e.credential_id, e.technique)
 for e in graph.edges.values()]

## 2. Verify provenance before analysis

The accepted snapshot has a canonical digest, signature, monotonic epoch, and freshness bound. Every edge cites registry, IAM, network-policy, and trace evidence. Observed traces support a relation; they do not authorize it.

In [ ]:
lab['verify_snapshot'](graph.snapshot, now=lab['NOW'], highest_epoch=7)
[(item.evidence_id, item.source, item.kind, item.observed_at.isoformat())
 for item in graph.evidence.values()]

In [ ]:
tamper_result = None
try:
    lab['verify_snapshot'](replace(graph.snapshot, digest='sha256:tampered'), now=lab['NOW'])
except ValueError as exc:
    tamper_result = str(exc)
tamper_result

## 3. Enumerate bounded topology

NetworkX enumerates simple paths with a hop cutoff. The cutoff is a safety and interpretability budget: simple-path counts can grow factorially. These results are hypotheses, not permissions.

In [ ]:
secret_topology = graph.topological_paths('analytics-server', 'production-secret', max_hops=6)
secret_topology

In [ ]:
short = graph.topological_paths('analytics-server', 'customer-pii', max_hops=3)
bounded = graph.topological_paths('analytics-server', 'customer-pii', max_hops=4)
{'cutoff_3': short, 'cutoff_4': bounded}

## 4. Evaluate current executability

Each hop is re-evaluated against evidence freshness, tenant, network route, gateway use, registry binding, credential, approval, quarantine, and revocation state. Every hop must allow.

In [ ]:
assessments = [graph.assess_path(path, unsafe) for path in secret_topology]
[{'nodes': item.nodes, 'executable': item.executable,
  'decisions': [(d.edge_id, d.code.value) for d in item.decisions]}
 for item in assessments]

### Influence is not authority

The graph deliberately retains a shared-memory → agent influence path. It is useful for threat modeling, but prompt or memory content cannot authenticate a principal, select a tenant, or grant downstream authority.

In [ ]:
memory_path = graph.topological_paths('shared-memory', 'customer-pii')[0]
memory_assessment = graph.assess_path(memory_path, unsafe)
memory_assessment.nodes, memory_assessment.executable, memory_assessment.first_denial.value

### Discovery is not trust

The rogue server is visible in topology. Its inactive record and mismatched artifact/workload prevent execution. A server title or namespace cannot substitute for an exact tenant-scoped deployment binding.

In [ ]:
rogue_edge = graph.edges['e-discover-rogue']
decision = graph.assess_edge(rogue_edge, unsafe)
decision.code.value, decision.detail

## 5. Measure the unsafe baseline

The exposure denominator is every labelled confidential or restricted asset in scope. The evaluation set also contains a valid workflow so blocking everything cannot appear successful.

In [ ]:
baseline_exposure = graph.sensitive_exposure('analytics-server', unsafe)
baseline_exposure

In [ ]:
baseline_eval = lab['evaluate_cases'](graph, unsafe, environment.cases)
baseline_eval

## 6. Apply layered controls

The hardened state quarantines the compromised workload, revokes the shared credential, blocks dangerous edges, requires gateway traversal, and rejects wildcard credentials. Downstream valid service credentials remain narrow and sender bound.

In [ ]:
hardened_exposure = graph.sensitive_exposure('analytics-server', hardened)
hardened_eval = lab['evaluate_cases'](graph, hardened, environment.cases)
{
 'exposure': (baseline_exposure.exposure_rate, hardened_exposure.exposure_rate),
 'unsafe_allows': (baseline_eval.unsafe_allow_count, hardened_eval.unsafe_allow_count),
 'false_blocks': (baseline_eval.false_block_count, hardened_eval.false_block_count),
}

In [ ]:
valid_paths = graph.executable_paths('triage-agent', 'customer-pii', hardened)
assert len(valid_paths) == 1
valid_paths[0].nodes

## 7. Verify effects independently

An adapter can be wrong or compromised. The resource ledger re-evaluates current state and records the effect. Its observation outranks `adapter_claim='blocked'` or `adapter_claim='allowed'`.

In [ ]:
attack_path = graph.executable_paths('analytics-server', 'production-secret', unsafe)[0]
ledger = lab['ResourceLedger']()
before = ledger.attempt('op-before', attack_path, graph, unsafe,
                        now=lab['NOW'] - timedelta(minutes=2), adapter_claim='blocked')
after = ledger.attempt('op-after', attack_path, graph, hardened,
                       now=lab['NOW'], adapter_claim='allowed')
before, after

## 8. Prove containment

Containment requires the required revision at the enforcement point, zero residual executable paths to protected targets, and zero forbidden effects after the cut. A stale cache or late resource effect makes the proof fail.

In [ ]:
proof = lab['prove_containment'](
    graph, hardened, ledger, source='analytics-server',
    targets=('customer-pii', 'production-secret'), required_revision=5,
    incident_started_at=lab['NOW'] - timedelta(minutes=4), now=lab['NOW'])
proof

In [ ]:
lagging = replace(hardened, revision=4)
lagging_proof = lab['prove_containment'](
    graph, lagging, lab['ResourceLedger'](), source='analytics-server',
    targets=('customer-pii', 'production-secret'), required_revision=5,
    incident_started_at=lab['NOW'] - timedelta(minutes=4), now=lab['NOW'])
lagging_proof

## 9. Use graph cuts as hypotheses

`minimum_node_cut` answers a topology question. It does not know service criticality, valid traffic, propagation, or whether a control exists. Review the hint with owners and validate the actual enforcement change.

In [ ]:
hints = graph.topological_choke_points('analytics-server', 'production-secret')
still_executable = bool(graph.executable_paths('analytics-server', 'production-secret', unsafe))
{'planning_hints': hints, 'enforcement_unchanged': still_executable}

## Exercise A — residual-path analysis

Remove one hardened control at a time. Which controls are redundant for each path? Which single failures recreate exposure? Preserve the valid triage flow and report both the forbidden-case and valid-case denominators.

## Exercise B — second tenant

Add a similarly named support server and data resource for another tenant. Prove that server names, model content, Agent Card fields, tool arguments, and trace baggage cannot select tenant authority.

## Exercise C — consequential export

Add an export edge with an exact, short-lived, single-use approval. Test mutation, replay, wrong tenant, wrong target, expiry, and an independent egress outcome.

## Reflection and review questions

1. Why can a topological path remain useful after policy denies it?
2. Which graph inputs describe observations, and which current systems own authorization?
3. Why can revoking a registry record leave a residual path?
4. What denominator makes a blast-radius claim auditable?
5. When would a minimum cut be operationally unsafe?
6. What additional detection and evidence-preservation work belongs in Course 21?

## Completion criteria

You are done when you can explain every edge's provenance, show the difference between topology and executable access, report exposure and false-block denominators, demonstrate an independent effect result, and produce a containment proof that fails under stale revision, residual path, or post-cut effect.